## Setup and Imports

In [ ]:
# Standard Library Imports
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Add src directory to path
project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Project utilities
from utils.helpers import load_orders, set_style, save_figure

set_style()

print("All imports successful!")
print(f"Project root: {project_root}")
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

## Load Data

In [ ]:
processed_path = project_root / 'data' / 'processed'
input_file = processed_path / 'orders_with_anomaly_flag.csv'

df_raw = pd.read_csv(input_file)
df = df_raw.copy()

# Parse timestamps
df['PURCHASE_TS'] = pd.to_datetime(df['PURCHASE_TS'], errors='coerce')
df['SHIP_TS'] = pd.to_datetime(df['SHIP_TS'], errors='coerce')

# Extract date components
df['PURCHASE_YEAR'] = df['PURCHASE_TS'].dt.year
df['PURCHASE_MONTH'] = df['PURCHASE_TS'].dt.month
df['PURCHASE_YEARMONTH'] = df['PURCHASE_TS'].dt.to_period('M')

print(f'Loaded: {len(df):,} rows, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
print(f'\nIS_ANOMALY breakdown:\n{df["IS_ANOMALY"].value_counts()}')

## Overall Price Distribution

In [ ]:
# Basic price statistics across dataset

print('Overall price statistics')
price_stats = df['USD_PRICE'].describe()
print(f" Count:   {price_stats['count']:>10,.0f}")
print(f" Mean:    {price_stats['mean']:>10,.2f}")
print(f" Std Dev: {price_stats['std']:>10,.2f}")
print(f" Min:     {price_stats['min']:>10,.2f}")
print(f" 25%:     {price_stats['25%']:>10,.2f}")
print(f" Median:  {price_stats['50%']:>10,.2f}")
print(f" 75%:     {price_stats['75%']:>10,.2f}")
print(f" Max:     {price_stats['max']:>10,.2f}")
print()

# Check for skewness
# A large gap between mean and median suggests skewness, which is common in price data due to outliers.
# If the mean is significantly higher than the median, it indicates a right-skewed distribution, often caused by a few very high-priced orders.
# Conversely, if the mean is lower than the median, it suggests a left-skewed distribution, which is less common in price data.
mean = price_stats['mean']
median = price_stats['50%']
gap = mean - median
print(f"Mean vs Median gap: ${gap:.2f}")
if gap > 0:
    print(f"The mean is higher than the median, indicating a right-skewed distribution likely due to high-priced outliers.")
elif gap < 0:
    print(f"The median is higher than the mean, indicating a left-skewed distribution, which is less common in price data.")
else:
    print(f"The mean and median are equal, suggesting a symmetric distribution, which is unusual for price data and may indicate data issues.")

In [ ]:

print(f"Price Points of Interest:")
zero_prices = df[df['USD_PRICE'] == 0]
null_prices = df[df['USD_PRICE'].isnull()]
under_10    = df[df['USD_PRICE'] < 10]
over_1000   = df[df['USD_PRICE'] > 1000]
over_2000   = df[df['USD_PRICE'] > 2000]

print(f" Exactly $0:          {len(zero_prices):>6,} orders ({len(zero_prices)/len(df)*100:.2f}%)")
print(f" Null/missing prices: {len(null_prices):>6,} orders ({len(null_prices)/len(df)*100:.2f}%)")
print(f" Under $10:           {len(under_10):>6,} orders ({len(under_10)/len(df)*100:.2f}%)")
print(f" Over $1,000:         {len(over_1000):>6,} orders ({len(over_1000)/len(df)*100:.2f}%)")
print(f" Over $2,000:         {len(over_2000):>6,} orders ({len(over_2000)/len(df)*100:.2f}%)")

## Extreme Value Deep Dive

In [ ]:
# Calculate IQR and outlier thresholds for dataset
Q1 = df['USD_PRICE'].quantile(0.25) # 25th percentile
Q3 = df['USD_PRICE'].quantile(0.75) # 75th percentile
IQR = Q3 - Q1 # Interquartile Range

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

print(f"Price outlier thresholds:")
print(f' Q1 (25th percentile): ${Q1:.2f}')
print(f' Q3 (75th percentile): ${Q3:.2f}')
print(f' IQR:                  ${IQR:.2f}')
print(f' Lower bound:          ${lower_bound:.2f}')
print(f' Upper bound:          ${upper_bound:.2f}')
print()

low_outliers = df[df['USD_PRICE'] < lower_bound]
high_outliers = df[df['USD_PRICE'] > upper_bound]

print(f' Low outliers  (< ${lower_bound:.2f}): {len(low_outliers):,}')
print(f' High outliers (> ${upper_bound:.2f}): {len(high_outliers):,}')

In [ ]:
# Investigate zero and null prices
# Are they concentrated in a specific product, country or channel? 
# This can help identify if the issue is related to specific SKUs, regions, 
# or sales channels, which may point to data entry errors, promotional pricing, or other anomalies.

print('$0 Price Orders Investigation')
print(f'Total $0 price orders: {len(zero_prices):,}')
print()

print('Product drilldown:')
print(zero_prices['PRODUCT_ID'].value_counts().to_string())
print()

print('By marketing channel:')
print(zero_prices['MARKETING_CHANNEL'].value_counts().to_string())
print()

print('By purchase platform:')
print(zero_prices['PURCHASE_PLATFORM'].value_counts().to_string())
print()

print('By country(top 10):')
print(zero_prices['COUNTRY_CODE'].value_counts().head(10).to_string())
print()

In [ ]:
# Investigate high price outliers
# Are these real premium orders, or could they be data errors?

print('High Price Outliers Investigation')
print(f'Total high price outliers (> ${upper_bound:.2f}): {len(high_outliers):,}')
print()

print('Top 20 most expensive orders:')
top_prices = df.nlargest(20, 'USD_PRICE')[[
    'PRODUCT_NAME', 'USD_PRICE', 'COUNTRY_CODE', 
    'MARKETING_CHANNEL', 'PURCHASE_PLATFORM', 'PURCHASE_TS'
]]
print(top_prices.to_string(index=False))
print()

print('High price orders by product:')
print(high_outliers['PRODUCT_NAME'].value_counts().to_string())
print()

## Price Variation by Product

In [ ]:
# Calculate per-product price statistics to 
# identify if certain products have unusually high or low average prices, which could indicate pricing errors or misclassifications.

print('Per-product price statistics:')
product_price_stats = df.groupby('PRODUCT_NAME')['USD_PRICE'].agg([
    ('count',   'count'),
    ('mean',    'mean'),
    ('median',  'median'),
    ('std',     'std'),
    ('min',     'min'),
    ('max',     'max'),
    ('25%',     lambda x: x.quantile(0.25)),
    ('75%',     lambda x: x.quantile(0.75)),
]).round(2) # pyright: ignore[reportArgumentType, reportCallIssue]

# Calclulate IQR and outlier thresholds for each product

product_price_stats['IQR']         = product_price_stats['75%'] - product_price_stats['25%']
product_price_stats['lower_bound'] = product_price_stats['25%'] - (1.5 * product_price_stats['IQR'])
product_price_stats['upper_bound'] = product_price_stats['75%'] + (1.5 * product_price_stats['IQR'])
product_price_stats['price_range'] = product_price_stats['max'] - product_price_stats['min']

print(product_price_stats[['count', 'median', 'min', 'max', 'price_range', 'lower_bound', 'upper_bound']].to_string())

In [ ]:
# Flag per-product outliers using groupby and transform

# Calculate IQR and outlier thresholds for each product
# transform will return a series the same legth as df so we can create new columns to flag outliers

df['PRICE_Q1'] = df.groupby('PRODUCT_NAME')['USD_PRICE'].transform(lambda x: x.quantile(0.25))
df['PRICE_Q3'] = df.groupby('PRODUCT_NAME')['USD_PRICE'].transform(lambda x: x.quantile(0.75))
df['PRICE_IQR'] = df['PRICE_Q3'] - df['PRICE_Q1']

# Calculate lower and upper bounds for each product

df['PRICE_LOWER_BOUND'] = df['PRICE_Q1'] - (1.5 * df['PRICE_IQR'])
df['PRICE_UPPER_BOUND'] = df['PRICE_Q3'] + (1.5 * df['PRICE_IQR'])

# Flag outliers

df['IS_PRICE_OUTLIER'] = (
    (df['USD_PRICE'] < df['PRICE_LOWER_BOUND']) | 
    (df['USD_PRICE'] > df['PRICE_UPPER_BOUND'])
)

# Separatly flag $0 price orders

df['IS_ZERO_PRICE'] = df['USD_PRICE'] == 0

print('Per_Product outlier flags added:')
print(f" Total price outliers flagged:          {df['IS_PRICE_OUTLIER'].sum():,}")
print(f" Total $0 price orders flagged:         {df['IS_ZERO_PRICE'].sum():,}")
print()

print('Outlier count per product:')
outlier_by_product = df.groupby('PRODUCT_NAME')['IS_PRICE_OUTLIER'].agg(['sum', 'mean']).round(3)
outlier_by_product.columns = ['outlier_count', 'outlier_rate']
outlier_by_product['outlier_rate'] = (outlier_by_product['outlier_rate'] * 100).round(2) # Convert to percentage
print(outlier_by_product.sort_values('outlier_rate', ascending=False).to_string())

## Price Variation by Country, Channel and Time

In [ ]:
# Price by country (top 15)
# Use median price to mitigate skew from outliers and get a more representative view of typical order values in each country.

print('Price by country (top 15)')
top_countries = df['COUNTRY_CODE'].value_counts().head(15).index
country_price_stats = df[df['COUNTRY_CODE'].isin(top_countries)].groupby('COUNTRY_CODE')['USD_PRICE'].agg([
      ('orders',        'count'),
    ('median_price',  'median'),
    ('mean_price',    'mean'),
    ('min_price',     'min'),
    ('max_price',     'max'),
]).round(2).sort_values('median_price', ascending=False) # pyright: ignore[reportArgumentType, reportCallIssue]

print(country_price_stats.to_string())

In [ ]:
# Price by marketing channel
# Affiliate channels may have lower average prices due to discounts or promotions, while direct channels might show higher prices.

print('Median price by marketing channel')
channel_price_stats = df.groupby('MARKETING_CHANNEL')['USD_PRICE'].agg([
    ('orders',        'count'),
    ('median_price',  'median'),
    ('mean_price',    'mean'),
    ('min_price',     'min'),
    ('max_price',     'max'),
]).round(2).sort_values('median_price', ascending=False) # pyright: ignore[reportArgumentType, reportCallIssue]

print(channel_price_stats.to_string())

In [ ]:
# Price trend over time
# Analyze how prices have evolved over time to identify any unusual spikes or drops that could indicate data
# entry errors, seasonal effects, or the impact of promotions.

print('Monthly median price trend over time')
monthly_price_trend = df.groupby('PURCHASE_YEARMONTH')['USD_PRICE'].agg([
    ('orders',        'count'),
    ('median_price',  'median'),
    ('mean_price',    'mean'),
]).round(2) # pyright: ignore[reportArgumentType, reportCallIssue]

print(monthly_price_trend.to_string())

In [ ]:
# Price by product per year
# Has the average price of certain products changed significantly over time, 
# which could indicate pricing errors or shifts in product mix?

print('Price by product per year')
product_yearly_price = df.groupby(['PRODUCT_NAME', 'PURCHASE_YEAR'])['USD_PRICE'].median().round(2).unstack(fill_value=0) # pyright: ignore[reportArgumentType, reportCallIssue]
print(product_yearly_price.to_string())

## Visualisations

In [ ]:
# Chart 1: Overall price distribution
# Two charts side by side:
# Left: full range and shows outliers
# Right: capped at $600 — shows the main distribution more clearly

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['USD_PRICE'].dropna(), bins=100,
             color='#2E75B6', alpha=0.8, edgecolor='white')
axes[0].set_title('Full price distribution\n(including outliers)', fontsize=12)
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Number of orders')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))

axes[1].hist(df[df['USD_PRICE'] <= 600]['USD_PRICE'], bins=60,
             color='#2E75B6', alpha=0.8, edgecolor='white')
axes[1].set_title('Price distribution capped at $600\n(main distribution visible)', fontsize=12)
axes[1].set_xlabel('Price (USD)')
axes[1].set_ylabel('Number of orders')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))

fig.suptitle('USD Price Distribution — GameZone Orders',
             fontsize=14, fontweight='medium', y=1.02)
save_figure(fig, '02_price_distribution_overview.png')
plt.show()

In [ ]:
# Chart 2: Box plot by product
# A box plot is the best chart for showing price spread across categories
# The box = IQR (middle 50% of values)
# The line inside = median
# The whiskers = fences (1.5 × IQR)
# The dots = outliers beyond the fences

fig, ax = plt.subplots(figsize=(14, 7))

# Sort products by median price for a clean ordered chart
product_order = df.groupby('PRODUCT_NAME')['USD_PRICE'].median().sort_values().index

sns.boxplot(
    data=df,
    x='USD_PRICE',
    y='PRODUCT_NAME',
    order=product_order,
    color='#2E75B6',
    flierprops=dict(marker='o', markerfacecolor='#C00000',
                    markersize=3, alpha=0.4),
    ax=ax
)

ax.set_title('Price distribution by product\nRed dots = outliers beyond 1.5 × IQR',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Price (USD)')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))

save_figure(fig, '02_price_boxplot_by_product.png')
plt.show()

In [ ]:
# Chart 3: Median price over time per product
# Shows whether prices are stable, rising, or falling across the dataset period
# Each line represents one product

fig, ax = plt.subplots(figsize=(14, 6))

monthly_by_product = df.groupby(
    ['PURCHASE_YEARMONTH', 'PRODUCT_NAME']
)['USD_PRICE'].median().reset_index()
monthly_by_product['PURCHASE_YEARMONTH'] = monthly_by_product['PURCHASE_YEARMONTH'].astype(str)

products = df['PRODUCT_NAME'].unique()
colours  = plt.cm.tab10.colors

for i, product in enumerate(products):
    subset = monthly_by_product[monthly_by_product['PRODUCT_NAME'] == product]
    ax.plot(subset['PURCHASE_YEARMONTH'], subset['USD_PRICE'],
            label=product, linewidth=1.5, color=colours[i % len(colours)])

ax.set_title('Median price per product over time',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Month')
ax.set_ylabel('Median price (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))
plt.xticks(rotation=45, ha='right')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)

save_figure(fig, '02_median_price_over_time.png')
plt.show()

In [ ]:
# Chart 4: Price outlier rate by product
# Which products have the most price instability?

fig, ax = plt.subplots(figsize=(12, 5))

outlier_plot = outlier_by_product.sort_values('outlier_rate')
bars = ax.barh(outlier_plot.index, outlier_plot['outlier_rate'],
               color='#C00000', alpha=0.8)

for bar, rate in zip(bars, outlier_plot['outlier_rate']):
    ax.text(bar.get_width() + 0.2,
            bar.get_y() + bar.get_height() / 2,
            f'{rate}%', va='center', fontsize=10)

ax.set_title('Price outlier rate by product\n(% of orders outside 1.5 × IQR fence)',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Outlier rate (%)')

save_figure(fig, '02_price_outlier_rate_by_product.png')
plt.show()

In [ ]:
# Chart 5: Median price by marketing channel
# Do certain channels consistently yield higher or lower order values?

fig, ax = plt.subplots(figsize=(10, 5))

channel_order = channel_price_stats.sort_values('median_price').index
bars = ax.barh(
    channel_order,
    channel_price_stats.loc[channel_order, 'median_price'],
    color='#2E75B6', alpha=0.85
)

for bar, val in zip(bars, channel_price_stats.loc[channel_order, 'median_price']):
    ax.text(bar.get_width() + 2,
            bar.get_y() + bar.get_height() / 2,
            f'${val:.0f}', va='center', fontsize=10)

overall_median = df['USD_PRICE'].median()
ax.axvline(overall_median, color='#C00000', linestyle='--', linewidth=1.5,
           label=f'Overall median: ${overall_median:.0f}')

ax.set_title('Median order value by marketing channel',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Median price (USD)')
ax.legend()

save_figure(fig, '02_median_price_by_channel.png')
plt.show()

## Findings and Recommendations

In [ ]:
print('=' * 65)
print('  NOTEBOOK 2 FINDINGS — PRICE FORENSICS')
print('=' * 65)
print()
print('FINDING 1 — Overall distribution is right-skewed')
print('  Median price: $168. Mean price: $281. The $113 gap between')
print('  them is caused entirely by high-priced outliers pulling the')
print('  mean upward — a classic sign of right skew. For any revenue')
print('  analysis, median is the more reliable central measure.')
print()
print('FINDING 2 — $0 orders are likely test or cancelled transactions')
print('  29 orders (0.13%) have a $0 price. All 29 are on the website')
print('  platform. 20 of 29 share the same product_id (f5ca), suggesting')
print('  a specific SKU recording issue rather than random data entry')
print('  errors. These should be excluded from all revenue analysis.')
print()
print('FINDING 3 — PS5 extreme prices are likely currency errors')
print('  The top 20 most expensive orders are ALL Sony PlayStation 5')
print('  Bundles, predominantly from GB (United Kingdom), recorded via')
print('  direct channel on website. Prices reaching $3,147 are not')
print('  credible USD prices for a PS5 — the retail price is ~$499.')
print('  The most likely explanation is GBP amounts recorded without')
print('  USD conversion. Additionally, the highest PS5 price appears')
print('  in May 2019 — over a year before the PS5 was announced (June')
print('  2020), confirming these records are anomalous.')
print()
print('FINDING 4 — High outlier rates on accessories are statistical')
print('  Dell Gaming Mouse (31.6%) and JBL Headset (12.4%) have the')
print('  highest outlier rates, but this reflects their very tight')
print('  normal price ranges ($48-$53 and $18-$27 respectively).')
print('  Even small price deviations exceed the IQR fence on low-cost')
print('  items. This is a statistical artefact, not a data quality crisis.')
print()
print('FINDING 5 — Channel has no influence on price')
print('  All channels share a median price of $168 except email ($159).')
print('  Pricing is entirely product-driven. This is important for')
print('  Project 02 CLV modelling — channel should not be used as a')
print('  price predictor.')
print()
print('FINDING 6 — Prices are stable over time')
print('  Monthly median price is $168 consistently across 2019-2021.')
print('  No seasonal pricing, no promotional discounts detectable at')
print('  the aggregate level. Product-level prices are also stable —')
print('  the Nintendo Switch holds exactly $168 throughout the period.')
print()
print('FINDING 7 — Duplicate product name confirmed again')
print('  27inches 4k gaming monitor appears as a separate row in the')
print('  per-product statistics with only 61 records vs 4,662 for the')
print('  canonical name. It also shows 0.00 median price in 2021 —')
print('  suggesting it stopped being recorded after a certain date.')
print('  This confirms it is a legacy name for the same product.')
print()
print('RECOMMENDATION — Price flags for downstream analysis')
print('  Three categories of records to handle carefully:')
print()
print('  Flag 1 — IS_ZERO_PRICE: 29 records at exactly $0')
print('           Action: EXCLUDE from all revenue and CLV analysis')
print()
print('  Flag 2 — IS_PRICE_OUTLIER: 1,430 records outside per-product')
print('           IQR fence. Action: RETAIN but flag. Exclude from')
print('           average price calculations, retain for order counts.')
print()
print('  Flag 3 — IS_CURRENCY_SUSPECT: PS5 orders from GB with price')
print('           above $2,000. Action: FLAG for manual review.')
print('           Likely GBP recorded as USD.')
print()
print('  The cleaned dataset exported from this notebook applies all')
print('  three flags. Downstream notebooks use IS_ZERO_PRICE to filter')
print('  and IS_PRICE_OUTLIER to conditionally exclude from averages.')
print()
print('=' * 65)

## Export to Processed Data

In [ ]:
# Drop the intermediate IQR calculation columns before saving
# These were working columns, not analytical outputs — no need to carry them forward
cols_to_drop = ['PRICE_Q1', 'PRICE_Q3', 'PRICE_IQR',
                'PRICE_LOWER_BOUND', 'PRICE_UPPER_BOUND']
df_out = df.drop(columns=cols_to_drop)

output_file = processed_path / 'orders_with_price_flags.csv'
df_out.to_csv(output_file, index=False)

print('Saved: data/processed/orders_with_price_flags.csv')
print(f'  Rows:    {len(df_out):,}')
print(f'  Columns: {list(df_out.columns)}')
print()

# Summary of all flags in the output file
print('Flag Summary in Output File')
print(f"  IS_ANOMALY (from NB1):      {df_out['IS_ANOMALY'].sum():,} records")
print(f"  IS_ZERO_PRICE:              {df_out['IS_ZERO_PRICE'].sum():,} records")
print(f"  IS_PRICE_OUTLIER:           {df_out['IS_PRICE_OUTLIER'].sum():,} records")
print()
print('Notebook 2 complete ✓')
print('Figures saved     → reports/figures/  (5 charts)')
print('Processed data    → data/processed/orders_with_price_flags.csv')
print('Ready for         → Notebook 3: Entity Resolution')